In [1]:
# Install the necessary libraries
!pip install -q torch transformers datasets accelerate

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset

# Check if the GPU is awake and ready
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Constructing on: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Constructing on: cuda
GPU Name: Tesla T4


In [3]:
# 1. Load the "Dummy Brain" (GPT-2)
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name).to(device)

# GPT-2 doesn't have a padding token by default, so we set one
tokenizer.pad_token = tokenizer.eos_token

# 2. Define the Ribosome Scorer
class RibosomeScorer(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # This is the "lightweight pass". We compress the hidden state,
        # find the patterns, and output a single float.
        self.scorer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(), # GELU is standard for modern transformers
            nn.Linear(hidden_size // 2, 1),
            nn.Sigmoid() # This forces the output to be strictly between 0.0 and 1.0
        )

    def forward(self, hidden_states):
        # Input shape: [Batch, Sequence_Length, Hidden_Size]
        # We pass it through our scorer
        scores = self.scorer(hidden_states)
        # Output shape is [Batch, Sequence, 1]. We squeeze it to [Batch, Sequence]
        return scores.squeeze(-1)

# Initialize our custom Ribosome and put it on the GPU
hidden_size = base_model.config.n_embd
ribosome = RibosomeScorer(hidden_size).to(device)

# 3. The Test Run
test_sentence = "The mechanism is absolutely revolutionary."
print(f"Testing sentence: '{test_sentence}'\n")

# Tokenize and push to GPU
inputs = tokenizer(test_sentence, return_tensors="pt").to(device)

# Tell PyTorch not to calculate gradients yet (saves memory during inference)
with torch.no_grad():
    # Get the raw contextual embeddings from the base model
    outputs = base_model(**inputs)
    hidden_states = outputs.last_hidden_state

    # Pass those embeddings into our Ribosome
    importance_scores = ribosome(hidden_states)

# 4. Print the Results
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
scores = importance_scores[0].cpu().numpy() # Pull back to CPU to print

print("--- Ribosome Importance Scores ---")
for token, score in zip(tokens, scores):
    # Clean up GPT-2's weird 'Ġ' character which represents a space
    clean_token = token.replace('Ġ', '')
    print(f"{clean_token:<15} : {score:.4f}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Testing sentence: 'The mechanism is absolutely revolutionary.'

--- Ribosome Importance Scores ---
The             : 0.6148
mechanism       : 0.6447
is              : 0.6251
absolutely      : 0.6376
revolutionary   : 0.5647
.               : 0.7627


In [4]:
import numpy as np

def assemble_metatokens(tokens, scores):
    # 1. Find the Peaks (Local Maxima)
    peaks = []
    for i in range(len(scores)):
        is_peak = True
        # Check left neighbor
        if i > 0 and scores[i] < scores[i-1]:
            is_peak = False
        # Check right neighbor
        if i < len(scores) - 1 and scores[i] < scores[i+1]:
            is_peak = False

        if is_peak:
            peaks.append(i)

    # Fallback just in case the wave is perfectly flat
    if not peaks:
        peaks.append(np.argmax(scores))

    # 2. Initialize Metatoken Buckets
    metatokens = {p: [] for p in peaks}

    # 3. The Gravity Pass (Assigning valleys to peaks)
    for i, (token, score) in enumerate(zip(tokens, scores)):
        if i in peaks:
            metatokens[i].append((i, token, score))
            continue

        # Find distances from this word to all peaks
        distances = {p: abs(i - p) for p in peaks}
        min_dist = min(distances.values())

        # Which peaks are closest?
        closest_peaks = [p for p, d in distances.items() if d == min_dist]

        if len(closest_peaks) == 1:
            # Gravity pulls it to the nearest peak
            metatokens[closest_peaks[0]].append((i, token, score))
        else:
            # TIE-BREAKER: Load Balancing
            # If a word is exactly between two peaks, send it to the lighter peak
            peak_weights = [scores[p] for p in closest_peaks]
            lightest_peak = closest_peaks[np.argmin(peak_weights)]
            metatokens[lightest_peak].append((i, token, score))

    # 4. Format and maintain Temporal Tags (Original Order)
    final_groups = []
    for p in sorted(metatokens.keys()):
        # Sort by their original index to keep the sequence of time intact
        group = sorted(metatokens[p], key=lambda x: x[0])

        group_tokens = [t[1].replace('Ġ', '') for t in group]
        additive_score = sum([t[2] for t in group])

        final_groups.append({
            "anchor_word": tokens[p].replace('Ġ', ''),
            "chunk": " ".join(group_tokens),
            "total_weight": additive_score
        })

    return final_groups

# Run our test data through the assembler
metatokens = assemble_metatokens(tokens, scores)

print("--- Metatoken Assembly (Untrained) ---")
for i, mt in enumerate(metatokens):
    print(f"\nMetatoken {i+1}:")
    print(f"  Anchor  : {mt['anchor_word']}")
    print(f"  Chunk   : [{mt['chunk']}]")
    print(f"  Weight  : {mt['total_weight']:.4f}")

--- Metatoken Assembly (Untrained) ---

Metatoken 1:
  Anchor  : mechanism
  Chunk   : [The mechanism]
  Weight  : 1.2595

Metatoken 2:
  Anchor  : absolutely
  Chunk   : [is absolutely revolutionary]
  Weight  : 1.8274

Metatoken 3:
  Anchor  : .
  Chunk   : [.]
  Weight  : 0.7627


In [5]:
print("--- The Constructor: Building Concept Vectors ---\n")

# We will store our new compressed "Idea Vectors" here
concept_vectors = []

# We need to map our text chunks back to their hidden states
current_idx = 0
for mt in metatokens:
    # Count how many tokens are in this specific chunk
    chunk_length = len(mt['chunk'].split())

    # Slice the raw GPU math (hidden states) for these specific tokens
    # hidden_states shape from our first cell is [1, Sequence_Length, 768]
    chunk_hidden_states = hidden_states[0, current_idx : current_idx + chunk_length, :]

    # COMPRESSION: Average the word vectors together into ONE single "Concept Vector"
    concept_vector = torch.mean(chunk_hidden_states, dim=0)

    concept_vectors.append({
        "chunk": mt['chunk'],
        "weight": mt['total_weight'],
        "vector": concept_vector,
        "temporal_tag": current_idx # Remembering the original order
    })

    # Move the index forward for the next chunk
    current_idx += chunk_length

# THE CASCADE: Sort by Weight (Heaviest / Most Important First)
priority_queue = sorted(concept_vectors, key=lambda x: x['weight'], reverse=True)

print("Priority Processing Order (Heaviest to Lightest):")
for i, concept in enumerate(priority_queue):
    print(f"{i+1}. Ingesting Concept: [{concept['chunk']}] (Weight: {concept['weight']:.4f})")
    print(f"   -> Compressed to Latent Vector of shape: {list(concept['vector'].shape)}")

--- The Constructor: Building Concept Vectors ---

Priority Processing Order (Heaviest to Lightest):
1. Ingesting Concept: [is absolutely revolutionary] (Weight: 1.8274)
   -> Compressed to Latent Vector of shape: [768]
2. Ingesting Concept: [The mechanism] (Weight: 1.2595)
   -> Compressed to Latent Vector of shape: [768]
3. Ingesting Concept: [.] (Weight: 0.7627)
   -> Compressed to Latent Vector of shape: [768]


In [6]:
import torch.nn.functional as F

print("--- The Decoder: Unfolding and Articulating ---\n")

# 1. The Skeleton: Re-establishing Time
# We take our priority concepts and put them back in their original temporal order
temporal_queue = sorted(concept_vectors, key=lambda x: x['temporal_tag'])

# We pull GPT-2's entire vocabulary embedding matrix (Shape: 50,257 words x 768 dimensions)
# This is our mathematical "dictionary"
vocab_embeddings = base_model.get_input_embeddings().weight

print("Reconstructing Output from Latent Concepts:\n")

for i, concept in enumerate(temporal_queue):
    print(f"Concept {i+1} (Original Group: [{concept['chunk']}]):")

    # 2. Translating the Vector back to Words
    # We take our compressed concept and compare it against every word in the dictionary
    vector = concept['vector'].unsqueeze(0) # Format for cosine similarity

    # Calculate how mathematically similar our chunk is to every single word GPT-2 knows
    similarities = F.cosine_similarity(vector, vocab_embeddings)

    # Get the top 3 closest matching words
    top_k = 3
    top_scores, top_indices = torch.topk(similarities, top_k)

    print("  -> Closest Latent Meanings in Vocabulary:")
    for score, idx in zip(top_scores, top_indices):
        # Decode the token ID back to English text
        token = tokenizer.decode(idx).strip()
        token = token if token else "[EMPTY/SPECIAL]"
        print(f"       {token:<15} (Match Score: {score:.4f})")
    print("-" * 50)

--- The Decoder: Unfolding and Articulating ---

Reconstructing Output from Latent Concepts:

Concept 1 (Original Group: [The mechanism]):
  -> Closest Latent Meanings in Vocabulary:
       SPONSORED       (Match Score: -0.0796)
       soDeliveryDate  (Match Score: -0.0846)
       theless         (Match Score: -0.0851)
--------------------------------------------------
Concept 2 (Original Group: [is absolutely revolutionary]):
  -> Closest Latent Meanings in Vocabulary:
       SPONSORED       (Match Score: -0.0773)
       soDeliveryDate  (Match Score: -0.0816)
       theless         (Match Score: -0.0825)
--------------------------------------------------
Concept 3 (Original Group: [.]):
  -> Closest Latent Meanings in Vocabulary:
       SPONSORED       (Match Score: -0.0807)
       enegger         (Match Score: -0.0854)
       soDeliveryDate  (Match Score: -0.0856)
--------------------------------------------------


In [7]:
from datasets import load_dataset

print("--- Downloading Training Data ---")
# Using a small, high-quality text dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1%]")

# We need to chunk the raw text into manageable pieces (e.g., 64 tokens)
def tokenize_function(examples):
    # Truncate and pad so all sequences are exactly the same length for the GPU
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

# Clean out empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

print("Tokenizing dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Convert to PyTorch tensors
tokenized_datasets.set_format("torch")
print(f"Dataset ready! Sample size: {len(tokenized_datasets)} batches.")

--- Downloading Training Data ---


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/367 [00:00<?, ? examples/s]

Tokenizing dataset...


Map:   0%|          | 0/258 [00:00<?, ? examples/s]

Dataset ready! Sample size: 258 batches.


In [8]:
class RibosomeCascadeTrainerModel(nn.Module):
    def __init__(self, base_model, ribosome):
        super().__init__()
        self.base_model = base_model
        self.ribosome = ribosome

        # We add a Linear layer at the end to map our 768-D vectors back to the 50k vocabulary
        self.lm_head = nn.Linear(base_model.config.n_embd, base_model.config.vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None, labels=None):
        # 1. Get raw embeddings
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        # 2. Ribosome Pass (Get boundaries)
        # Note: In a real training loop, we use "soft" boundaries (probabilities)
        # so backpropagation can actually flow through the math.
        importance_scores = self.ribosome(hidden_states)

        # 3. Simulate the Cascade (Weighted Attention)
        # We multiply the hidden states by their importance scores.
        # This forces the model to pay attention to high-score words and ignore low-score words.
        weighted_states = hidden_states * importance_scores.unsqueeze(-1)

        # 4. Predict the words (The Decoder)
        logits = self.lm_head(weighted_states)

        loss = None
        if labels is not None:
            # Shift so that tokens < n predict n
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            # Reconstruction Loss (Did it get the words right?)
            loss_fct = nn.CrossEntropyLoss()
            recon_loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

            # Sparsity Penalty (Did it make too many peaks?)
            # We penalize the Ribosome if the average importance score is too high.
            sparsity_loss = torch.mean(importance_scores)

            # Combine them: 90% focus on accuracy, 10% focus on compression
            loss = recon_loss + (0.1 * sparsity_loss)

        return {"loss": loss, "logits": logits}

# Initialize the combined model
trainable_model = RibosomeCascadeTrainerModel(base_model, ribosome).to(device)
print("Unified model architecture ready for training.")

Unified model architecture ready for training.


In [9]:
from transformers import Trainer, TrainingArguments

print("--- Configuring the Training Loop ---")

# Set up the Training Arguments
# We are keeping this lightweight so it executes quickly on the Colab GPU
training_args = TrainingArguments(
    output_dir="./ribosome_checkpoints",
    num_train_epochs=1,               # Just 1 pass over the data for the PoC
    per_device_train_batch_size=8,    # 8 sentences at a time
    learning_rate=5e-5,               # Standard learning rate
    logging_steps=10,                 # Print the loss every 10 steps so we can watch it learn
    remove_unused_columns=False,      # CRITICAL: Forces HF not to throw away our custom inputs
    report_to="none"                  # Disables weights & biases logging for a clean local run
)

# Initialize the Hugging Face Trainer
trainer = Trainer(
    model=trainable_model,
    args=training_args,
    train_dataset=tokenized_datasets
)

--- Configuring the Training Loop ---
Starting the Ribosome training loop. Watch the loss go down!



TypeError: unsupported operand type(s) for /: 'NoneType' and 'int'

In [10]:
# 1. The Fix (PyTorch Edition)
def add_labels(examples):
    # PyTorch uses .clone() instead of standard Python .copy()
    examples["labels"] = examples["input_ids"].clone()
    return examples

print("Adding labels to the dataset...")
tokenized_datasets = tokenized_datasets.map(add_labels, batched=True)

# 2. Re-initialize the Trainer with the fixed dataset
trainer = Trainer(
    model=trainable_model,
    args=training_args,
    train_dataset=tokenized_datasets
)

# 3. Take Three!
print("Take three! Starting the Ribosome training loop...")
trainer.train()

Adding labels to the dataset...


Map:   0%|          | 0/258 [00:00<?, ? examples/s]

Take three! Starting the Ribosome training loop...


Step,Training Loss
10,12.226485
20,10.422354
30,10.273283


TrainOutput(global_step=33, training_loss=10.901742646188447, metrics={'train_runtime': 24.8136, 'train_samples_per_second': 10.398, 'train_steps_per_second': 1.33, 'total_flos': 0.0, 'train_loss': 10.901742646188447, 'epoch': 1.0})

In [11]:
# 1. Set the model to Evaluation Mode (turns off training behaviors like dropout)
trainable_model.eval()

test_sentence = "The mechanism is absolutely revolutionary."
print(f"Testing TRAINED sentence: '{test_sentence}'\n")

# Tokenize and push to GPU
inputs = tokenizer(test_sentence, return_tensors="pt").to(device)

# Tell PyTorch we are just doing inference, no backprop
with torch.no_grad():
    # Get the raw contextual embeddings from the base model
    outputs = trainable_model.base_model(**inputs)
    hidden_states = outputs.last_hidden_state

    # Pass those embeddings into our NEWLY TRAINED Ribosome!
    trained_importance_scores = trainable_model.ribosome(hidden_states)

# Format and Print the Results
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
scores = trained_importance_scores[0].cpu().numpy()

print("--- Ribosome Importance Scores (TRAINED) ---")
for token, score in zip(tokens, scores):
    clean_token = token.replace('Ġ', '')
    print(f"{clean_token:<15} : {score:.4f}")

Testing TRAINED sentence: 'The mechanism is absolutely revolutionary.'

--- Ribosome Importance Scores (TRAINED) ---
The             : 0.3675
mechanism       : 0.2262
is              : 0.2094
absolutely      : 0.1628
revolutionary   : 0.1977
.               : 0.1752


In [12]:
import torch
import os
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

print("--- The Deep Dive: Full Dataset Training ---")

# 1. Load the FULL training split this time
print("Downloading full Wikitext-2 dataset...")
full_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# Clean empty lines
full_dataset = full_dataset.filter(lambda x: len(x["text"].strip()) > 0)

# 2. Tokenize and add labels (The ground truth for the loss function)
def tokenize_and_label(examples):
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )
    # PyTorch clone for labels
    tokenized["labels"] = [torch.tensor(ids).clone() for ids in tokenized["input_ids"]]
    return tokenized

print("Tokenizing... (This might take a minute)")
tokenized_full_dataset = full_dataset.map(tokenize_and_label, batched=True, remove_columns=["text"])
tokenized_full_dataset.set_format("torch")

print(f"Full Dataset Ready: {len(tokenized_full_dataset)} sequences.")

# 3. Configure the Long-Run Training Arguments
output_dir = "./ribosome_full_run"
long_run_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,               # 3 full passes over the data
    per_device_train_batch_size=16,   # Doubled the batch size to push the T4 GPU harder
    learning_rate=3e-5,               # Slightly lower learning rate for stability over time
    logging_steps=100,                # Log every 100 steps so you have a clean history to read
    save_strategy="epoch",            # Save a backup at the end of every epoch
    remove_unused_columns=False,
    report_to="none"
)

# 4. Re-initialize the Trainer with the big data
deep_trainer = Trainer(
    model=trainable_model,
    args=long_run_args,
    train_dataset=tokenized_full_dataset
)



# 5. Execute and Save
print("\nStarting the endurance run. See you when you get back!")
deep_trainer.train()

# 6. Save the final weights so they don't disappear
final_save_path = "./ribosome_final_weights"
trainable_model.save_pretrained(final_save_path) if hasattr(trainable_model, 'save_pretrained') else torch.save(trainable_model.state_dict(), os.path.join(final_save_path, "pytorch_model.bin"))
print(f"\nDone! Model weights safely stored in {final_save_path}")

--- The Deep Dive: Full Dataset Training ---


Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Tokenizing... (This might take a minute)


Map:   0%|          | 0/23767 [00:00<?, ? examples/s]

Full Dataset Ready: 23767 sequences.

Starting the endurance run. See you when you get back!


Step,Training Loss
100,6.815502
200,5.118068
300,4.705740
400,4.629926
500,4.599371
600,4.491254
700,4.521542
800,4.354479
900,4.238571
1000,4.119821


RuntimeError: Parent directory ./ribosome_final_weights does not exist.

In [13]:
import os
import torch

# The folder we want to save into
final_save_path = "./ribosome_final_weights"

# THE FIX: Tell Python to actually create the folder first!
os.makedirs(final_save_path, exist_ok=True)

# Now we can safely save the weights
if hasattr(trainable_model, 'save_pretrained'):
    trainable_model.save_pretrained(final_save_path)
else:
    torch.save(trainable_model.state_dict(), os.path.join(final_save_path, "pytorch_model.bin"))

print(f"Done! Model weights safely stored in {final_save_path}")

Done! Model weights safely stored in ./ribosome_final_weights


In [14]:
# 1. Ensure the model is strictly in Evaluation Mode
trainable_model.eval()

# Let's test a complex sentence with a mix of heavy concepts and fluffy filler
test_sentence = "Although the initial dataset was completely chaotic, the new neural architecture processed the latent patterns effortlessly."
print(f"Target Sentence: '{test_sentence}'\n")

# Tokenize and push to GPU
inputs = tokenizer(test_sentence, return_tensors="pt").to(device)

# 2. Inference pass (No gradients, just pure forward thinking)
with torch.no_grad():
    # Get the raw contextual embeddings
    outputs = trainable_model.base_model(**inputs)
    hidden_states = outputs.last_hidden_state

    # Let the trained Ribosome score the sequence
    trained_importance_scores = trainable_model.ribosome(hidden_states)

# Format for output
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
scores = trained_importance_scores[0].cpu().numpy()

print("--- Step 1: Trained Ribosome Scores ---")
for token, score in zip(tokens, scores):
    clean_token = token.replace('Ġ', '')
    print(f"{clean_token:<15} : {score:.4f}")

print("\n--- Step 2: The Metatoken Assembly (Wave & Gravity) ---")
# We reuse the wave/gravity function we built earlier!
trained_metatokens = assemble_metatokens(tokens, scores)

for i, mt in enumerate(trained_metatokens):
    print(f"Metatoken {i+1}:")
    print(f"  Anchor (Peak) : {mt['anchor_word']}")
    print(f"  Chunk         : [{mt['chunk']}]")
    print(f"  Total Weight  : {mt['total_weight']:.4f}\n")

Target Sentence: 'Although the initial dataset was completely chaotic, the new neural architecture processed the latent patterns effortlessly.'

--- Step 1: Trained Ribosome Scores ---
Although        : 0.9778
the             : 0.9888
initial         : 0.9751
dataset         : 0.9917
was             : 0.9953
completely      : 0.9779
chaotic         : 0.9905
,               : 0.9466
the             : 0.9834
new             : 0.9693
neural          : 0.9527
architecture    : 0.9804
processed       : 0.9634
the             : 0.9519
latent          : 0.9150
patterns        : 0.9799
effortlessly    : 0.9760
.               : 0.9451

--- Step 2: The Metatoken Assembly (Wave & Gravity) ---
Metatoken 1:
  Anchor (Peak) : the
  Chunk         : [Although the initial]
  Total Weight  : 2.9417

Metatoken 2:
  Anchor (Peak) : was
  Chunk         : [dataset was]
  Total Weight  : 1.9870

Metatoken 3:
  Anchor (Peak) : chaotic
  Chunk         : [completely chaotic]
  Total Weight  : 1.9685

Metatoken